# Setup

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


In [2]:
# Import primary libraries
import sklearn
import xgboost
import lightgbm as lgb
import numpy as np
import pandas as pd

# Import secondary libraries
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [3]:
# Load the competition datasets
df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

# Exploratory Data Analysis (EDA)

In [4]:
# Inspect first five rows of the train dataset
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [5]:
# Inspect first five rows of the test dataset
test_df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


In [6]:
# Examine the dataset's structure, column data types, and non-null values
print("Info:\n", df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

In [7]:
# Examine the distribution of the target variable: Will_Buy_EV
df["Will_Buy_EV"].value_counts(normalize=True)

Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64

In [8]:
# Separate the numerical and categorical feature columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
if "Will_Buy_EV" in cat_cols:
    cat_cols.remove("Will_Buy_EV")

# Data Preprocessing

In [9]:
# Map the target variable to numeric values by converting string columns to category dtype
evmap = {'No': 0, 'Yes': 1}
df["Will_Buy_EV"] = df["Will_Buy_EV"].map(evmap).fillna(-1)

In [10]:
def create_interaction_features(df):
    df = df.copy()

    # Binary Helpers & Numerical Conversions
    subsidy_yes = (df['Subsidy_Available'] == 'Yes').astype(int)
    no_subsidy = (df['Subsidy_Available'] == 'No').astype(int)
    home_charge_yes = (df['Home_Charging_Possible'] == 'Yes').astype(int)

    # Map Range Anxiety Level to Numeric
    anxiety_map = {'Low': 1, 'Medium': 2, 'High': 3}
    if df['Range_Anxiety_Level'].dtype == 'object' or df['Range_Anxiety_Level'].dtype.name == 'category':
        df['Anxiety_Num'] = df['Range_Anxiety_Level'].astype(str).map(anxiety_map).fillna(1)
    else:
        df['Anxiety_Num'] = df['Range_Anxiety_Level'].fillna(1)

    # 1. Primary Interaction Features (Subsidies & Triggers)
    # High Impact: Subsidy + Environmental Concern
    df['Env_x_Subsidy'] = df['Environmental_Concern_Level'] * subsidy_yes
    
    # High Impact: Income + Subsidy
    df['Income_x_Subsidy'] = df['Annual_Income_USD'] * subsidy_yes
    
    # Purchase power when NO subsidy is available
    df['No_Subsidy_Purchase_Power'] = no_subsidy * (df['Annual_Income_USD'] / 10000.0) * df['Environmental_Concern_Level']

    # 2. Charging Infrastructure Features
    # Total Near Charging Stations
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    
    # Overall Infrastructure Score (Home charging given higher weight)
    df['Total_Charge_Access'] = df['Total_Charging_Stations'] + (home_charge_yes * 2)

    # 3. Anxiety, Commute & Ratio Interactions
    # Anxiety vs Infrastructure Access
    df['Anxiety_to_Charge_Ratio'] = df['Anxiety_Num'] / (df['Total_Charge_Access'] + 1e-5)
    
    # Income to Anxiety Ratio
    df['Income_to_Anxiety_Ratio'] = df['Annual_Income_USD'] / (df['Anxiety_Num'] + 1e-5)
    
    # Daily Commute vs Total Charging Stations
    df['Commute_per_Charging_Station'] = df['Daily_Commute_km'] / (df['Total_Charging_Stations'] + 1e-5)
    
    # Anxiety Override Score
    df['Anxiety_Override_Score'] = (df['Daily_Commute_km'] * df['Environmental_Concern_Level']) / (df['Anxiety_Num'] + 1e-5)

    return df

# Apply train function
df = create_interaction_features(df)
test_df = create_interaction_features(test_df)

In [11]:
for col in cat_cols:
    df[col] = df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

In [12]:
x = df.drop(['id', 'Will_Buy_EV'], axis=1)
y = df['Will_Buy_EV']
x_test = test_df.drop(['id'], axis=1)

# Stratified K-Fold Cross-Validation & Model Training

In [13]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(len(df))
test_pred = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(x, y)):
    print(f"\n>>>>>> FOLD {fold+1}<<<<<<")
    x_train, y_train = x.iloc[train_idx], y.iloc[train_idx]
    x_val, y_val = x.iloc[val_idx], y.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=3500,
        learning_rate=0.007,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        early_stopping_rounds=150,
        random_state=42
    )

    model.fit(
        x_train, y_train,
        eval_set=[(x_val, y_val)],
        verbose=500
    )

    val_pred = model.predict_proba(x_val)[:,1]
    oof_pred[val_idx] = val_pred
    test_pred += model.predict_proba(x_test)[:,1] / skf.n_splits
    fold_auc = roc_auc_score(y_val, val_pred)
    print(f'---Fold {fold+1} ROC-AUC: {fold_auc:.4f}---')

print("\n" + "=" * 40)
print(f'---Overall ROC/AUC Score: {fold_auc:.4f}---')
print("\n" + "=" * 40)


>>>>>> FOLD 1<<<<<<
[0]	validation_0-logloss:0.45975


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [04:56:42] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [04:56:42] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[500]	validation_0-logloss:0.23343
[1000]	validation_0-logloss:0.23025
[1500]	validation_0-logloss:0.22935
[2000]	validation_0-logloss:0.22877
[2500]	validation_0-logloss:0.22845
[3000]	validation_0-logloss:0.22827
[3499]	validation_0-logloss:0.22814
---Fold 1 ROC-AUC: 0.9409---

>>>>>> FOLD 2<<<<<<
[0]	validation_0-logloss:0.45973


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:00:26] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:00:26] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[500]	validation_0-logloss:0.23159
[1000]	validation_0-logloss:0.22850
[1500]	validation_0-logloss:0.22768
[2000]	validation_0-logloss:0.22715
[2500]	validation_0-logloss:0.22685
[3000]	validation_0-logloss:0.22667
[3499]	validation_0-logloss:0.22657
---Fold 2 ROC-AUC: 0.9418---

>>>>>> FOLD 3<<<<<<
[0]	validation_0-logloss:0.45973


...

[500]	validation_0-logloss:0.22975
[1000]	validation_0-logloss:0.22640
[1500]	validation_0-logloss:0.22547
[2000]	validation_0-logloss:0.22491
[2500]	validation_0-logloss:0.22458
[3000]	validation_0-logloss:0.22439
[3499]	validation_0-logloss:0.22429
---Fold 3 ROC-AUC: 0.9431---

>>>>>> FOLD 4<<<<<<
[0]	validation_0-logloss:0.45973


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:07:46] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:07:46] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[500]	validation_0-logloss:0.23024
[1000]	validation_0-logloss:0.22701
[1500]	validation_0-logloss:0.22624
[2000]	validation_0-logloss:0.22574
[2500]	validation_0-logloss:0.22548
[3000]	validation_0-logloss:0.22535
[3499]	validation_0-logloss:0.22524
---Fold 4 ROC-AUC: 0.9425---

>>>>>> FOLD 5<<<<<<
[0]	validation_0-logloss:0.45975


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:11:25] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:11:25] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[500]	validation_0-logloss:0.23137
[1000]	validation_0-logloss:0.22813
[1500]	validation_0-logloss:0.22728
[2000]	validation_0-logloss:0.22672
[2500]	validation_0-logloss:0.22641
[3000]	validation_0-logloss:0.22623
[3499]	validation_0-logloss:0.22616
---Fold 5 ROC-AUC: 0.9420---

---Overall ROC/AUC Score: 0.9420---



In [14]:
oof_predl = np.zeros(len(df))
test_predl = np.zeros(len(test_df))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(skf.split(x,y)):
    print(f"\n>>>>>FOLD {fold+1}<<<<<")

    x_trainl,y_trainl = x.iloc[train_idx],y.iloc[train_idx]
    x_vall, y_vall = x.iloc[val_idx], y.iloc[val_idx]

    modell = LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.007,
        num_leaves=31,
        random_state=42
    )

    modell.fit(
        x_trainl, y_trainl,
        eval_set=[(x_vall, y_vall)],
        callbacks=[lgb.early_stopping(stopping_rounds=200)]
    )

    val_predl = modell.predict_proba(x_vall)[:,1]
    oof_predl[val_idx] = val_predl
    test_predl += modell.predict_proba(x_test)[:,1] / skf.n_splits

    fold_aucl = roc_auc_score(y_vall, val_predl)
    print(f'--- Fold {fold + 1} ROC-AUC: {fold_aucl:.4f} ---')

print("\n" + "=" * 40)
print(f'Overall OOF ROC-AUC Score: {roc_auc_score(y, oof_predl):.4f}')
print("=" * 40)


>>>>>FOLD 1<<<<<
[LightGBM] [Info] Number of positive: 93424, number of negative: 441508
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037202 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2071
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174646 -> initscore=-1.553048
[LightGBM] [Info] Start training from score -1.553048
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[4974]	valid_0's binary_logloss: 0.228424
--- Fold 1 ROC-AUC: 0.9407 ---

>>>>>FOLD 2<<<<<
[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027872 seconds.
You can set `force_row_wise=true` to remove t

# File Submission

In [15]:
# Combined Out-of-Fold Predictions
final_oof = (oof_pred * 0.5) + (oof_predl * 0.5)
print(f'Ensemble OOF ROC-AUC Score: {roc_auc_score(y, final_oof):.4f}')

# Combined Test Predictions
final_test_pred = (test_pred * 0.5) + (test_predl * 0.5)
print(f'Test Prediction Rate: {final_test_pred}')

# Generate a submission file
submission = pd.DataFrame({'id': test_df["id"], 'Will_Buy_EV': final_test_pred})
submission.to_csv("submission_2026-9-28", index=False)
print("Successfully saved as CSV file.")

Ensemble OOF ROC-AUC Score: 0.9422
Test Prediction Rate: [0.0085431  0.02192536 0.00599105 ... 0.00306654 0.00043518 0.00030528]
Successfully saved as CSV file.


In [16]:
submission.head()

,id,Will_Buy_EV
0,668665,0.008543
1,668666,0.021925
2,668667,0.005991
3,668668,0.002864
4,668669,0.018267
